# Core Concepts: Wavefronts and Optical Elements

Wavefronts provide a mathematical description of how light propagates through space. Understanding wavefronts is essential for modeling optical systems, from simple lenses to complex astronomical instruments. This tutorial introduces the `Wavefront` object and demonstrates how optical elements transform wavefronts through amplitude and phase modifications.

**Why wavefronts matter:**

Wavefronts describe the complete state of monochromatic light at any point in an optical system. Unlike simple ray tracing, wavefront methods capture diffraction, interference, and aberration effects—phenomena critical for high-resolution imaging and wavefront sensing applications. The wavefront representation enables accurate calculation of point spread functions (PSFs), modulation transfer functions (MTFs), and optical transfer functions (OTFs), which characterize imaging system performance.

**Topics covered:**
* Wavefront representation and physical interpretation
* Fraunhofer and Fresnel propagation regimes
* Common optical elements and their effects
* Building sequential optical systems

In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt

## Wavefronts

A `Wavefront` represents the spatial distribution of a monochromatic electromagnetic field. It stores the complex electric field amplitude—which contains both magnitude (intensity) and phase information—sampled on a grid.

**Physical interpretation:**

The electric field is represented as a complex number at each grid point: $E(x,y) = A(x,y) \cdot e^{i\phi(x,y)}$, where $A$ is the amplitude and $\phi$ is the phase. The amplitude relates to the intensity ($I = |E|^2$), while the phase contains information about the wave's propagation direction and optical path differences. Phase variations across a wavefront determine how the light will diffract and focus.

**Wavefront creation:**

To create a wavefront, you specify the complex electric field distribution and the wavelength. The field amplitude determines the intensity distribution, while the phase distribution determines the wavefront shape.

In [ ]:
# Create a grid and aperture
pupil_diameter = 1
pupil_grid = make_pupil_grid(128, pupil_diameter * 1.2)
aperture = make_circular_aperture(1.0)(pupil_grid)

# Create a wavefront with uniform phase (planar wavefront)
wavelength = 500e-9
wf = Wavefront(aperture, wavelength)

print(f"Wavelength: {wf.wavelength*1e9:.0f} nm")
print(f"Total power: {wf.total_power:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
imshow_field(wf.amplitude, ax=axes[0])
axes[0].set_title('Amplitude')
imshow_field(wf.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title('Phase')
imshow_field(wf.intensity, ax=axes[2])
axes[2].set_title('Intensity')
plt.show()

**Phase aberrations:**

Real optical systems introduce phase distortions called aberrations. These aberrations modify the wavefront shape and degrade image quality. Common aberrations include defocus (parabolic phase), astigmatism (cylindrical phase variations), coma, and spherical aberration.

The phase structure determines the interference pattern when the wavefront propagates. Even small phase variations (fractions of a wavelength) can significantly affect the resulting intensity distribution in the focal plane. This sensitivity is exploited in adaptive optics systems to measure and correct wavefront errors in real-time.

In [ ]:
# Defocus: parabolic phase error
# This simulates being out of focus by introducing a quadratic phase
defocus_phase = zernike(2, 0)(pupil_grid)
wf_defocus = Wavefront(aperture * np.exp(1j * defocus_phase), wavelength)

# Astigmatism: phase varies differently in x and y directions
# Creates two orthogonal focal lines instead of a point
astig_phase = zernike(2, 2)(pupil_grid)
wf_astig = Wavefront(aperture * np.exp(1j * astig_phase), wavelength)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imshow_field(wf_defocus.phase, cmap='RdBu', ax=axes[0])
axes[0].set_title('Defocus Phase')
imshow_field(wf_astig.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title('Astigmatism Phase')
plt.show()

## Propagation

Optical propagation calculates how wavefronts evolve as light travels through space. The propagation regime depends on the distance traveled relative to the wavelength and aperture size.

**Propagation regimes:**

**Fraunhofer (far-field) diffraction** applies when the propagation distance $z$ satisfies $z \gg D^2/\lambda$, where $D$ is the aperture diameter and $\lambda$ is the wavelength. In this regime, the field distribution at distance $z$ is the Fourier transform of the initial field. This is the standard regime for calculating focal plane PSFs in imaging systems.

**Fresnel (near-field) diffraction** applies at shorter distances where $z \sim D^2/\lambda$. This regime accounts for the quadratic phase factor that accumulates during propagation. Fresnel propagation is essential for modeling beam propagation, optical cavities, and near-field imaging systems.

**Fraunhofer propagation example:**

The Fraunhofer propagator computes the far-field diffraction pattern. In an imaging system, this corresponds to the focal plane intensity distribution (the PSF).

In [ ]:
wf_in = Wavefront(aperture, wavelength)

# Create a focal grid for the PSF calculation
# q=4 samples at 4x the Nyquist rate
# num_airy=20 covers 20 Airy rings
focal_grid = make_focal_grid(q=4, num_airy=20, spatial_resolution=wavelength / pupil_diameter)

# Fraunhofer propagator: pupil grid -> focal grid
fraunhofer = FraunhoferPropagator(pupil_grid, focal_grid)
wf_focal = fraunhofer(wf_in)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imshow_field(wf_in.amplitude, ax=axes[0])
axes[0].set_title('Pupil Plane Amplitude')

# Display log scale to see the diffraction rings
img = wf_focal.intensity / wf_focal.intensity.max()
imshow_field(np.log10(img + 1e-20), cmap='inferno', vmin=-5, ax=axes[1])
axes[1].set_title('Focal Plane PSF (Airy Pattern)')
plt.show()

**Fresnel propagation example:**

The Fresnel propagator includes the quadratic phase factor that develops during propagation. This is important when modeling beam propagation over moderate distances or when the observation plane is not in the far field.

In [ ]:
# Propagate over a finite distance (near-field regime)
distance = 10000.0  # meters
fresnel = FresnelPropagator(pupil_grid, distance)
wf_near = fresnel(wf_in)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
imshow_field(wf_near.intensity, cmap='inferno', ax=axes[0])
axes[0].set_title(f'Intensity at {distance}m')
imshow_field(wf_near.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title(f'Phase at {distance}m')
plt.show()

print(f"Fresnel number: {1.0**2 / (wavelength * distance):.2f}")

## Optical Elements

Optical elements modify wavefronts through amplitude and phase transformations. In HCIPy, optical elements are objects that can be applied to wavefronts to model physical optical components.

**Types of optical elements:**

* **Amplitude masks** attenuate light intensity (neutral density filters, apertures, coronagraph masks)
* **Phase screens** introduce phase delays (lenses, atmospheric turbulence, aberrations)
* **Propagators** move wavefronts between planes (Fraunhofer, Fresnel, angular spectrum)
* **Compound elements** combine multiple transformations

**Effect of phase aberrations on imaging:**

Phase errors in the pupil plane spread light away from the ideal focus, reducing peak intensity and creating a halo around the PSF. This demonstrates why wavefront quality is critical for high-contrast imaging.

In [ ]:
# Create a phase screen representing aberration or turbulence
# This is a simple sinusoidal phase pattern with Gaussian envelope
phase_screen = 0.5 * np.sin(2 * np.pi * pupil_grid.x) * np.exp(-pupil_grid.as_('polar').r**2)

# Apply the phase screen to the wavefront
wf_aberrated = Wavefront(wf_in.electric_field * np.exp(1j * phase_screen), wavelength)

# Propagate both ideal and aberrated wavefronts to compare
wf_focal_ideal = fraunhofer(wf_in)
wf_focal_aberrated = fraunhofer(wf_aberrated)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

imshow_field(phase_screen, cmap='RdBu', ax=axes[0,0])
axes[0,0].set_title('Phase Screen (radians)')

imshow_field(np.log10(wf_focal_ideal.intensity / wf_focal.intensity.max() + 1e-20), cmap='inferno', vmin=-5, ax=axes[0, 1])
axes[0,1].set_title('Ideal PSF')

imshow_field(wf_aberrated.phase, cmap='RdBu', ax=axes[1,0])
axes[1,0].set_title('Aberrated Wavefront Phase')

imshow_field(np.log10(wf_focal_aberrated.intensity / wf_focal_aberrated.intensity.max() + 1e-20), cmap='inferno', vmin=-5, ax=axes[1, 1])
axes[1,1].set_title('Aberrated PSF')

plt.tight_layout()
plt.show()

# Calculate Strehl ratio (measure of wavefront quality)
strehl = wf_focal_aberrated.intensity.max() / wf_focal_ideal.intensity.max()
print(f"Strehl ratio: {strehl:.4f}")

## Optical Systems

Complex optical systems are built by chaining optical elements sequentially. Each element transforms the wavefront, and the output of one element becomes the input to the next.

**System modeling approach:**

Optical systems can be modeled as a pipeline of transformations. For example, a simple imaging system consists of:
1. Initial wavefront (collimated beam)
2. Aperture (limits the beam)
3. Lens (adds focusing phase)
4. Propagation to focal plane

This modular approach allows complex systems to be built from simple components. HCIPy's optical element classes support this by providing a consistent interface where each element acts as a callable that transforms a wavefront.

**Lens phase pattern:**

A thin lens introduces a phase delay that varies quadratically with radial position. This phase curvature converts planar wavefronts into spherical wavefronts that converge to a focus. The phase pattern is given by $\phi(r) = -\pi r^2 / (\lambda f)$, where $f$ is the focal length.

In [ ]:
# Build a simple optical system: collimated beam -> lens -> focus
focal_length = 10.0  # meters

# Lens phase: quadratic phase that focuses light
# The negative sign indicates convergence to a focus
lens_phase = -np.pi * (pupil_grid.x**2 + pupil_grid.y**2) / (wavelength * focal_length)

# Apply lens to the wavefront
wf_after_lens = Wavefront(wf_in.electric_field * np.exp(1j * lens_phase), wavelength)

# Propagate to focal plane (Fraunhofer is appropriate for focal plane)
wf_focus = fraunhofer(wf_after_lens)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Phase added by lens
imshow_field(lens_phase, cmap='RdBu', ax=axes[0])
axes[0].set_title('Lens Phase Pattern')

# PSF at focus
imshow_field(np.log10(wf_focus.intensity + 1e-20), vmin=-5, ax=axes[1])
axes[1].set_title('Focal Plane Intensity')

# Phase at focus (shows spherical wave converging)
imshow_field(wf_focus.phase, cmap='RdBu', ax=axes[2])
axes[2].set_title('Phase at Focus')

plt.show()

print(f"Focal length: {focal_length} m")
print(f"f-number: {focal_length / 1.0:.1f}")  # Diameter is 1.0 from aperture

## Summary

This tutorial covered the fundamental concepts of wavefronts and optical elements in HCIPy.

**Wavefronts**
* Represent the complete state of monochromatic light as a complex electric field
* Contain both amplitude (intensity) and phase information
* Phase variations determine diffraction and focusing behavior
* Aberrations manifest as phase errors that degrade image quality

**Propagation**
* **Fraunhofer propagation** for far-field calculations (focal planes, distant screens)
* **Fresnel propagation** for near-field calculations (beam propagation, moderate distances)
* Choice of propagator depends on the Fresnel number and physical geometry

**Optical Elements**
* Transform wavefronts through amplitude or phase modifications
* Can be chained sequentially to model complex optical systems
* Phase screens represent aberrations, turbulence, and optical components

**Key Applications**
* Calculating PSFs for imaging system design
* Modeling wavefront sensors and adaptive optics
* Simulating coronagraphs for exoplanet detection
* Analyzing effects of optical aberrations on image quality